In [1]:
!pip install groq -q
print("Libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.5 MB/s eta 0:00:00
Libraries installed successfully


In [2]:
import sqlite3
import pandas as pd
import os

from groq import Groq
import re

print("All Libraries imported successfully")

All Libraries imported successfully


In [3]:
import os
os.environ["GROQ_API_KEY"]=""
client=Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL="llama-3.1-8b-instant"

print("Groq client initialized successfully")
print(f"Using model: {MODEL}")

Groq client initialized successfully
Using model: llama-3.1-8b-instant


In [4]:
import io

csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""

df=pd.read_csv(io.StringIO(csv_data))

print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
print("\nFirst 5 rows")
df.head()


Dataset loaded: 30 rows, 8 columns

First 5 rows


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [5]:
conn=sqlite3.connect("college.db")

df.to_sql("students",conn,if_exists="replace",index=False)

print("Database created: college.db")
print("Table 'students' created with 30 student records")

test_df=pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students",conn)
print(f"\nVerification: {test_df['total_rows'][0]} rows in database")

Database created: college.db
Table 'students' created with 30 student records

Verification: 30 rows in database


In [6]:
def get_schema(conn, table_name="students"):

  cursor=conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns=cursor.fetchall()

  schema_lines=[f"Table: {table_name}"]
  schema_lines.append("Columns:")

  for col in columns:
    schema_lines.append(f"  - {col[1]} ({col[2]})")

  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows=cursor.fetchall()
  schema_lines.append("\nSample rows (first 3):")

  for row in sample_rows:
    schema_lines.append(f"   {row}")

  return "\n".join(schema_lines)

schema=get_schema(conn)
print(schema)

Table: students
Columns:
  - student_id (INTEGER)
  - name (TEXT)
  - age (INTEGER)
  - gender (TEXT)
  - subject (TEXT)
  - marks (INTEGER)
  - attendance (INTEGER)
  - grade (TEXT)

Sample rows (first 3):
   (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
   (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
   (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [7]:
def generate_sql(user_question,schema_text,client,model):
  system_prompt = f"""You are an expert SQL assistant.
You are connected to a SQLite database with the following structure:


{schema_text}


Rules you must follow:
1. Generate ONLY a valid SQLite SQL query.
2. Do not include any explanation or text — only the SQL query.
3. Do not use markdown code blocks. Return the raw SQL only.
4. The table name is: students
5. Only use column names that exist in the schema above.
6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
"""
  response=client.chat.completions.create(
     model=model,
     messages=[
       {"role":"system","content":system_prompt},
       {"role":"user","content":user_question}
     ],

     temperature=0.0
   )

  sql_query=response.choices[0].message.content.strip()
  return sql_query


question="Show me all the male students whose grades are greater than C(I mean A and B)"
print(f"Question: {question}")
print("\nGenerating SQL...")
sql_query=generate_sql(question,schema,client,MODEL)
print(f"Generated SQL: {sql_query}")


Question: Show me all the male students whose grades are greater than C(I mean A and B)

Generating SQL...
Generated SQL: SELECT * FROM students WHERE gender = 'Male' AND grade IN ('A', 'B');


In [8]:
def execute_sql(sql_query, conn):
  clean_sql = sql_query.strip()
  clean_sql = re.sub(r'```sql\s*', '', clean_sql)
  clean_sql = re.sub(r'```s*', '', clean_sql)
  clean_sql = clean_sql.strip()
  try:
    result_df = pd.read_sql_query(clean_sql, conn)
    return result_df, None
  except Exception as e:
    return None, str(e)
print(f"Executing SQL: {sql_query}")
result, error = execute_sql(sql_query, conn)
if error:
  print(f"Error executing SQL: {error}")
else:
  print(f"\nQuery Returned {len(result)} rows:")
  print(result)

Executing SQL: SELECT * FROM students WHERE gender = 'Male' AND grade IN ('A', 'B');

Query Returned 10 rows:
   student_id           name  age gender      subject  marks  attendance grade
0           1   Aarav Sharma   20   Male  Mathematics     88          92     A
1           7    Karan Singh   22   Male  Mathematics     74          81     B
2           9   Vikram Reddy   20   Male      Science     70          79     B
3          15   Nikhil Verma   20   Male      Science     77          84     B
4          17   Manish Joshi   21   Male  Programming     73          82     B
5          19    Suresh Babu   22   Male  Mathematics     82          89     A
6          21    Deepak Nair   20   Male      Science     79          86     B
7          23   Sanjay Patel   21   Male  Programming     88          91     A
8          25    Vijay Kumar   22   Male  Mathematics     71          83     B
9          29  Ravi Krishnan   21   Male  Programming     78          88     B


In [9]:
def text_to_sql_agent(user_question, conn, client, model,verbose):

  print("="*60)

  print(f"USER QUESTION: {user_question}")
  print("="*60)

  if verbose:
    print("\n[STEP 1] Reading database schema...")
    schema_text=get_schema(conn)

  if verbose:
    print("Schema loaded successfully")

  if verbose:
    print("\n[STEP 2] Generating SQL query with Groq LLM...")

  generated_sql=generate_sql(user_question,schema_text,client,model)

  if verbose:
    print(f"Generated SQL:\n {generated_sql}")

  if verbose:
    print("\n[STEP 3] Executing SQL on the database...")

  result_df,error=execute_sql(generated_sql,conn)

  if error:
    print(f"SQL Execution Error: {error}")
    return None,generated_sql

  if verbose:
    print(f"\n[STEP 4] Query returned {len(result_df)} row(s)")
    print("\nRESULTS:")
    print("-"*40)
    print(result_df.to_string(index=False))

  print("="*60)

  return result_df,generated_sql

result,sql_used=text_to_sql_agent(
    "Show top 5 students in Programming",
    conn,client,MODEL,True
)


USER QUESTION: Show top 5 students in Programming

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT * FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
         11 Aditya Kumar   21   Male Programming     97          99    A+
          3  Rohan Mehta   20   Male Programming     95          98    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
          5   Arjun Nair   21   Male Programming     91          94    A+


In [10]:
result1, _ = text_to_sql_agent(
    "Show me all students who study Mathematics",
    conn,client,MODEL,True
)

USER QUESTION: Show me all students who study Mathematics

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT * FROM students WHERE subject = 'Mathematics'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 10 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 F

In [11]:
result2, _ = text_to_sql_agent(
    "Show me all students having age > 18",
    conn,client,MODEL,True
)

USER QUESTION: Show me all students having age > 18

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT * FROM students WHERE age > 18

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 30 row(s)

RESULTS:
----------------------------------------
 student_id           name  age gender     subject  marks  attendance grade
          1   Aarav Sharma   20   Male Mathematics     88          92     A
          2    Priya Patel   21 Female     Science     76          85     B
          3    Rohan Mehta   20   Male Programming     95          98    A+
          4     Sneha Iyer   22 Female Mathematics     62          78     C
          5     Arjun Nair   21   Male Programming     91          94    A+
          6 Divya Krishnan   20 Female     Science     83          88     A
          7    Karan Singh   22   Male Mathematics     74          81     B
          8   Ananya Gupta   21 Female Progra

In [12]:
result3, _ = text_to_sql_agent(
    "Count of all the students in maths and their marks",
    conn,client,MODEL,True
)

USER QUESTION: Count of all the students in maths and their marks

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT COUNT(student_id), SUM(marks) FROM students WHERE subject = 'Mathematics'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 1 row(s)

RESULTS:
----------------------------------------
 COUNT(student_id)  SUM(marks)
                10         735


In [13]:
result4, _ = text_to_sql_agent(
    "Show me all students who got A+ grade and display their names ",
    conn,client,MODEL,True
)

USER QUESTION: Show me all students who got A+ grade and display their names 

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT name FROM students WHERE grade = 'A+'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 7 row(s)

RESULTS:
----------------------------------------
         name
  Rohan Mehta
   Arjun Nair
 Aditya Kumar
Swathi Pillai
 Anjali Singh
  Nandita Rao
 Bhavna Mehta


In [14]:
!pip install crewai --force-reinstall litellm==1.30.0 --force-reinstall requests python-dotenv apscheduler --q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 976.4/976.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [15]:
import os

os.environ["GROQ_API_KEY"]=""

print("Groq api initialized succesfully")

Groq api initialized succesfully


In [16]:
!pip install 'crewai[litellm]' --upgrade --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 14.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you 

In [17]:
from crewai import Agent,Task, Crew

def research_agent_example():
  researcher= Agent(
    role='Research Analyst',
    goal='Identify top AI trends for 2025',
    backstory='Experienced technology researcher with 10+ years of industry experience',
    verbose=True,
    llm='groq/llama-3.3-70b-versatile'
)

  research_task=Task(
      description='List and analyze the top 3 AI trends for 2025 with examples',
      expected_output='Comprehensive report with detailed explanation for each trend',
      agent=researcher
  )

  crew=Crew(
      agents=[researcher],
      tasks=[research_task],
      verbose=True
  )

  return crew.kickoff()

result=research_agent_example()
print(result)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.6                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9a988068-dbd8-42c6-b927-f14c817f8f69                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: List and analyze the top 3 AI trends for 2025 with examples                                              │
│  ID: 03f05a17-6163-4379-965e-7bfa7af01c27                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Task: List and analyze the top 3 AI trends for 2025 with examples                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Comprehensive Report: Top 3 AI Trends for 2025**                                                             │
│                                                                                                                 │
│  As we approach 2025, the field of Artificial Intelligence (AI) is expected to witness significant              │
│  advancements, transforming various industries and aspects of our lives. Based on current developments and      │
│  industry forecasts, this report identifies and analyzes the top 3 AI trends for 2025, along with examples and  │
│  potential implications.                                                                                        │
│                                                                                                                 │
│  **Trend 1: Explainable AI (XAI) and Transparency**                                                             │
│                                                                                                                 │
│  Explainable AI (XAI) refers to the development of AI systems that provide transparent and interpretable        │
│  insights into their decision-making processes. This trend is driven by the need for accountability, trust,     │
│  and regulatory compliance in AI applications. XAI enables users to understand how AI models arrive at their    │
│  conclusions, fostering confidence in AI-driven decision-making.                                                │
│                                                                                                                 │
│  Examples of XAI applications include:                                                                          │
│                                                                                                                 │
│  * **Medical Diagnosis**: AI-powered diagnostic systems that provide explanations for their diagnoses,          │
│  allowing healthcare professionals to understand the reasoning behind the recommendations.                      │
│  * **Financial Services**: AI-driven credit scoring models that provide transparent explanations for credit     │
│  decisions, enabling consumers to understand the factors influencing their creditworthiness.                    │
│  * **Autonomous Vehicles**: XAI-enabled systems that provide insights into the decision-making processes of     │
│  self-driving cars, enhancing safety and trust in autonomous transportation.                                    │
│                                                                                                                 │
│  The adoption of XAI will have significant implications for various industries, including healthcare, finance,  │
│  and transportation. As AI becomes increasingly pervasive, the need for transparency and interpretability will  │
│  continue to grow, driving the development of XAI solutions.                                                    │
│                                                                                                                 │
│  **Trend 2: Edge AI and Distributed Intelligence**                                                              │
│                                                                                                                 │
│  Edge AI refers to the deployment of AI models on edge 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: List and analyze the top 3 AI trends for 2025 with examples                                              │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Comprehensive Report: Top 3 AI Trends for 2025**

As we approach 2025, the field of Artificial Intelligence (AI) is expected to witness significant advancements, transforming various industries and aspects of our lives. Based on current developments and industry forecasts, this report identifies and analyzes the top 3 AI trends for 2025, along with examples and potential implications.

**Trend 1: Explainable AI (XAI) and Transparency**

Explainable AI (XAI) refers to the development of AI systems that provide transparent and interpretable insights into their decision-making processes. This trend is driven by the need for accountability, trust, and regulatory compliance in AI applications. XAI enables users to understand how AI models arrive at their conclusions, fostering confidence in AI-driven decision-making.

Examples of XAI applications include:

* **Medical Diagnosis**: AI-powered diagnostic systems that provide explanations for their diagnoses, allowing healthcare professional